In [2]:
import langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Simple LLM call with streaming

In [16]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

In [6]:
model  = init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x160974d70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1609757f0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Create messages

In [17]:
messages = [
    SystemMessage("You are a good AI assistant."),
    HumanMessage("What are benefits of langchain."),
]

In [8]:
#invoke the model

response = model.invoke(messages)
response

AIMessage(content="LangChain is an open-source, Python-based framework for building large language models (LLMs) and other AI applications. It provides a set of tools and libraries to help developers create, train, and deploy LLMs more efficiently. The benefits of LangChain include:\n\n1. **Easy Model Integration**: LangChain simplifies the integration of LLMs into applications, making it easier to leverage the power of these models in various use cases.\n2. **Efficient Model Training**: LangChain provides pre-built tools and optimizers to streamline the process of training and fine-tuning LLMs, reducing the time and computational resources required.\n3. **Improved Model Performance**: LangChain's architecture and training techniques help to improve the performance of LLMs, enabling them to generate more accurate and informative responses.\n4. **Scalability**: LangChain is designed to handle large-scale model training and deployment, making it an ideal choice for applications that requ

In [11]:
#Streaming example

for chunk in model.stream(messages):
    print(chunk.content, end="",flush=True)

LangChain is an open-source framework for building large language models (LLMs) and multimodal AI applications. Some benefits of LangChain include:

1. **Easy Model Integration**: LangChain provides a simple interface for integrating various LLMs, such as LLaMA, BERT, and T5, into your applications.
2. **Modular Architecture**: LangChain's modular design allows you to easily swap out or add different models, libraries, and tools to your application as needed.
3. **Customizable Pipelines**: LangChain enables you to create custom pipelines for processing and generating text, images, and other data types.
4. **Improved Efficiency**: LangChain's framework allows for efficient utilization of computational resources, reducing the time and cost associated with training and deploying AI models.
5. **Scalability**: LangChain's design makes it easy to scale your applications to handle large volumes of data and complex tasks.
6. **Multimodal Support**: LangChain supports the integration of multip

In [21]:
#Dynamic prompt template
from langchain_core.prompts import ChatPromptTemplate


translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following text from {source_language} to {target_language}. Maintain the meaning and tone of the original text. {text}"),
    ("human", "{text}"),

])


prompt = translation_template.invoke({
    "source_language": "English",
    "target_language": "Spanish",
    "text": "Langchain makes AI development easier and more efficient."
})

In [22]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the following text from English to Spanish. Maintain the meaning and tone of the original text. Langchain makes AI development easier and more efficient.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes AI development easier and more efficient.', additional_kwargs={}, response_metadata={})])

In [23]:
translated_response = model.invoke(prompt)
print(translated_response)

content='Langchain facilita el desarrollo de inteligencia artificial de manera más fácil y eficiente.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 80, 'total_tokens': 99, 'completion_time': 0.021102848, 'completion_tokens_details': None, 'prompt_time': 0.004901809, 'prompt_tokens_details': None, 'queue_time': 0.051524021, 'total_time': 0.026004657}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e536c-4449-78b1-8b7e-e31b3e45b8d1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 19, 'total_tokens': 99}


### Build first chain

In [31]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

#template for story generation
def create_story_chain():
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short story based on the following prompt"),
        ("user", "Theme: {theme}\n Main Character:{main_character}\n Setting: {setting}"),
    ])

    #template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are literally a critic. Analyze the following story:"),
        ("user", "{story}"),
    ])

    story_chain = (
        story_prompt | model | StrOutputParser()
    )



    def analyze_story(story_text):
        return {"story":story_text}


    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt 
        | model
        | StrOutputParser()

    )

    return analysis_chain


In [32]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short story based on the following prompt'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main Character:{main_character}\n Setting: {setting}'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x160974d70>, async_client=<groq.resources.chat.comp

In [33]:
result = chain.invoke({
    "theme":"Artificial Intelligence",
    "main_character":"A curious robot named R1",
    "setting":"A futuristic city."
})

print("Story and Analysis")
print(result)
      

Story and Analysis
**A Critical Analysis of "The Curious Robot of Neo-Tokyo"**

**Strengths:**

1. **Engaging premise**: The story has a captivating premise, introducing a curious robot named R1 in a futuristic metropolis like Neo-Tokyo. This setup allows for exploration of themes like artificial intelligence, human connection, and self-discovery.
2. **Clear structure**: The narrative follows a logical progression, from R1's initial exploration to his discovery of the NeuroSpark and subsequent transformation. This structure makes the story easy to follow and understand.
3. **Well-developed characters**: R1, the protagonist, is a likable and relatable character. His curiosity and eagerness to learn make him endearing to readers. The old man, the shopkeeper, is also well-developed, serving as a mentor figure who guides R1 on his journey.
4. **Themes and symbolism**: The story explores themes like the potential of artificial intelligence, the importance of human connection, and the limita